# lab_02 — rozwiązania

## Klaster

In [ ]:
from dask.distributed import Client

client = Client(n_workers=2, threads_per_worker=2, memory_limit="3GiB")
client


In [ ]:
client.scheduler_info()["workers"].keys()

## Ćwiczenie NDVI

In [ ]:
import numpy as np
import dask.array as da
import time

shape = (365, 4, 3000, 3000)
dtype = np.float32
print("potrzebne GiB:", np.prod(shape) * np.dtype(dtype).itemsize / 1024**3)

try:
    arr_numpy = np.empty(shape, dtype=dtype)
    print("numpy OK")
except Exception as e:
    print("numpy OOM:", type(e).__name__, e)

chunks = (5, 4, 1000, 1000)
sat_data = da.random.uniform(0.0, 1.0, size=shape, chunks=chunks).astype(dtype)
sat_data


In [ ]:
red = sat_data[:, 2, :, :]
nir = sat_data[:, 3, :, :]
ndvi = (nir - red) / (nir + red + 1e-6)
print(ndvi.shape, ndvi.nbytes / 1024**3, "GiB")
ndvi


In [ ]:
t0 = time.perf_counter()
ndvi_time_series = ndvi.mean(axis=(1, 2)).compute()
print("sredni NDVI / dzien:", time.perf_counter() - t0, "s", ndvi_time_series.shape)

t0 = time.perf_counter()
drought_ratio = (ndvi < 0.2).mean().compute()
print("susza:", drought_ratio, "czas:", time.perf_counter() - t0, "s")


### Wnioski
- na dashboardzie pamiec workera skacze wokol rozmiaru chunka, nie rosnie do 52 GB
- `ndvi.compute()` bez redukcji probowaloby sciagnac ~13 GB do klienta (OOM)


## Zadanie 1

In [ ]:
import os
import numpy as np
import pandas as pd
import dask.dataframe as dd

DATA = os.path.join("data", "zamowienia.csv")
MISSING = os.path.join("data", "zamowienia_missing.csv")

pdf = pd.read_csv(DATA, sep=";")
print(pdf.dtypes)

rng = np.random.default_rng(42)
rows = rng.choice(np.arange(10, len(pdf)), size=8, replace=False)
cols = ["Kraj", "Sprzedawca", "Utarg", "idZamowienia"]
for r, c in zip(rows, rng.choice(cols, size=len(rows))):
    pdf.loc[r, c] = np.nan

pdf.to_csv(MISSING, sep=";", index=False)
print("NaN w wierszach:", sorted(rows.tolist()))


In [ ]:
ddf = dd.read_csv(MISSING, sep=";", assume_missing=True)
print(ddf.dtypes)
print("sum Utarg:", ddf["Utarg"].sum().compute())


In [ ]:
for n in [2, 10, 50, 200]:
    ddf_s = dd.read_csv(MISSING, sep=";", sample_rows=n, assume_missing=True)
    print("sample_rows=", n)
    print(ddf_s.dtypes)
    print()


## Zadanie 2

In [ ]:
# pip install dask-labextension
# jupyter server extension enable dask_labextension
print(client.dashboard_link)


## Zadanie 3

In [ ]:
import time
from pathlib import Path

INSTA = str(Path("data") / "instagram" / "*.parquet")
print(list(Path("data/instagram").glob("*.parquet")))

t0 = time.perf_counter()
ddf_raw = dd.read_parquet(INSTA)
n = len(ddf_raw)
_ = ddf_raw.head()
t_load = time.perf_counter() - t0

print("partycje:", ddf_raw.npartitions, "wiersze:", n, "czas:", t_load)
print(ddf_raw.dtypes)
ddf_raw


## Zadanie 4

In [ ]:
t0 = time.perf_counter()
top10 = ddf_raw.groupby("username")["likes"].sum().nlargest(10).compute()
t_top = time.perf_counter() - t0
print("czas:", t_top)
print(top10)


In [ ]:
t0 = time.perf_counter()
ddf_dates = ddf_raw.assign(
    date_parsed=dd.to_datetime(ddf_raw["date"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
)
h1_2019 = ddf_dates[
    (ddf_dates["date_parsed"] >= "2019-01-01") & (ddf_dates["date_parsed"] < "2019-07-01")
]
h1_count = len(h1_2019)
print("czas:", time.perf_counter() - t0, "wiersze:", h1_count)
h1_2019[["username", "date", "likes"]].head(10)


## Zadanie 5

In [ ]:
optimized_dtypes = {
    "sid": "int32",
    "sid_profile": "int32",
    "post_id": "string",
    "profile_id": "int64",
    "post_type": "int8",
    "description": "string",
    "likes": "int32",
    "comments": "int32",
    "username": "string",
    "bio": "string",
    "following": "int32",
    "followers": "int32",
    "num_posts": "int32",
    "is_business_account": "bool",
    "lang": "category",
    "category": "category",
}

t0 = time.perf_counter()
ddf_opt = dd.read_parquet(INSTA).astype(optimized_dtypes)
ddf_opt["date"] = dd.to_datetime(ddf_opt["date"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
n_opt = len(ddf_opt)
_ = ddf_opt.head()
t_load_opt = time.perf_counter() - t0

print("czas opt:", t_load_opt, "vs raw:", t_load)
print(ddf_opt.dtypes)


In [ ]:
t0 = time.perf_counter()
top10_opt = ddf_opt.groupby("username")["likes"].sum().nlargest(10).compute()
print("top10 opt:", time.perf_counter() - t0, "raw:", t_top)
print(top10_opt)

t0 = time.perf_counter()
h1_opt = ddf_opt[(ddf_opt["date"] >= "2019-01-01") & (ddf_opt["date"] < "2019-07-01")]
print("h1 opt:", time.perf_counter() - t0, "wiersze:", len(h1_opt))

ddf_opt.groupby("username")["likes"].sum().nlargest(10).optimize().pprint()


## Zadanie 6

In [ ]:
import statistics

SIZE = (20_000, 20_000)
N_RUNS = 10
chunk_configs = {
    "(2000,2000)": (2000, 2000),
    "(1000,1000)": (1000, 1000),
    "(4000,4000)": (4000, 4000),
    "(5000,5000)": (5000, 5000),
    "auto": None,
}

results = {}
for label, chunks in chunk_configs.items():
    times = []
    for _ in range(N_RUNS):
        if chunks is None:
            darr = da.random.normal(5, 0.2, size=SIZE)
        else:
            darr = da.random.normal(5, 0.2, size=SIZE, chunks=chunks)
        t0 = time.perf_counter()
        darr.mean(axis=0).compute()
        times.append(time.perf_counter() - t0)
    avg = statistics.mean(times)
    nch = int(np.prod([len(c) for c in darr.chunks]))
    results[label] = avg
    print(label, "avg=", avg, "chunks=", nch)

print("najszybsze:", min(results, key=results.get), results[min(results, key=results.get)])


Male chunki = duzo zadan i overhead. Duze = slabiej rownolegle i wiecej RAM.
Auto zwykle jest blisko optimum; czasem reczny podzial jest troche lepszy.


In [ ]:
# client.close()